# 03 — Feature Engineering & Pipeline de Preprocessing

🔒 **RÈGLE D'OR : le pipeline est fit() UNIQUEMENT sur France, puis transform() seul sur Cameroun.**
Ne jamais fit sur les données camerounaises — ce serait de la fuite d'information
qui invaliderait toute la validation externe.

Génère : Modèle A (12 variables) et Modèle B (Modèle A + niveau_etude).


In [1]:
import sys
sys.path.append('../src')
import pandas as pd
import numpy as np
import joblib
from feature_engineering import (
    MODEL_A_FEATURES, MODEL_A_NUMERIC, MODEL_A_CATEGORICAL,
    MODEL_B_FEATURES, MODEL_B_CATEGORICAL,
    build_preprocessing_pipeline, get_feature_names
)

fr = pd.read_csv('../data/processed/france_train_clean.csv')
cameroun = pd.read_csv('../data/processed/cameroun_combined_validation.csv')

print("France (train):", fr.shape)
print("Cameroun (validation externe):", cameroun.shape)
print(f"\nModèle A features ({len(MODEL_A_FEATURES)}):", MODEL_A_FEATURES)
print(f"Modèle B features ({len(MODEL_B_FEATURES)}):", MODEL_B_FEATURES)


France (train): (30000, 34)
Cameroun (validation externe): (455, 16)

Modèle A features (12): ['age_maternel', 'imc_ordinal', 'tension_ordinal', 'sa_premiere_consult', 'parite', 'atcd_familial_diabete_1er_deg', 'atcd_gdm', 'atcd_macrosomie', 'sopk', 'sedentarite', 'tabagisme', 'hta_ou_preeclampsie']
Modèle B features (13): ['age_maternel', 'imc_ordinal', 'tension_ordinal', 'sa_premiere_consult', 'parite', 'atcd_familial_diabete_1er_deg', 'atcd_gdm', 'atcd_macrosomie', 'sopk', 'sedentarite', 'tabagisme', 'hta_ou_preeclampsie', 'niveau_etude']


## 3.1 Préparation X / y — France (entraînement)

In [2]:
y_france = (fr['gdm_label'] == 'Oui').astype(int)

X_france_A = fr[MODEL_A_FEATURES].copy()
X_france_B = fr[MODEL_B_FEATURES].copy()

print(f"X_france_A: {X_france_A.shape}, taux positifs: {y_france.mean()*100:.1f}%")
X_france_A.isna().sum()


X_france_A: (30000, 12), taux positifs: 11.3%


age_maternel                        0
imc_ordinal                         0
tension_ordinal                  2322
sa_premiere_consult               887
parite                              0
atcd_familial_diabete_1er_deg    3576
atcd_gdm                         2371
atcd_macrosomie                  2964
sopk                             4490
sedentarite                      6593
tabagisme                        3046
hta_ou_preeclampsie                85
dtype: int64

## 3.2 Construction et FIT du pipeline (sur France uniquement)

In [3]:
# --- Pipeline Modèle A ---
preprocessor_A = build_preprocessing_pipeline(MODEL_A_NUMERIC, MODEL_A_CATEGORICAL, use_mice=True)
X_france_A_transformed = preprocessor_A.fit_transform(X_france_A)
print("Pipeline A fit sur France. Shape transformée:", X_france_A_transformed.shape)

# --- Pipeline Modèle B ---
preprocessor_B = build_preprocessing_pipeline(MODEL_A_NUMERIC, MODEL_B_CATEGORICAL, use_mice=True)
X_france_B_transformed = preprocessor_B.fit_transform(X_france_B)
print("Pipeline B fit sur France. Shape transformée:", X_france_B_transformed.shape)


Pipeline A fit sur France. Shape transformée: (30000, 22)
Pipeline B fit sur France. Shape transformée: (30000, 26)


## 3.3 TRANSFORM seul sur Cameroun (jamais de fit !)

In [4]:
X_cameroun_A = cameroun[MODEL_A_FEATURES].copy()
X_cameroun_B = cameroun[MODEL_B_FEATURES].copy()
y_cameroun = (cameroun['gdm_label'] == 'Oui').astype(int)

X_cameroun_A_transformed = preprocessor_A.transform(X_cameroun_A)  # transform() SEULEMENT
X_cameroun_B_transformed = preprocessor_B.transform(X_cameroun_B)

print("Cameroun transformé (Modèle A):", X_cameroun_A_transformed.shape)
print(f"Taux positifs Cameroun: {y_cameroun.mean()*100:.1f}% (rappel: 243/455 = 53.4%, structure différente de France)")


Cameroun transformé (Modèle A): (455, 22)
Taux positifs Cameroun: 53.4% (rappel: 243/455 = 53.4%, structure différente de France)


## 3.4 Sauvegarde pipelines + datasets transformés

In [5]:
joblib.dump(preprocessor_A, '../models/preprocessor_A.pkl')
joblib.dump(preprocessor_B, '../models/preprocessor_B.pkl')

np.save('../data/processed/X_france_A.npy', X_france_A_transformed)
np.save('../data/processed/X_france_B.npy', X_france_B_transformed)
np.save('../data/processed/X_cameroun_A.npy', X_cameroun_A_transformed)
np.save('../data/processed/X_cameroun_B.npy', X_cameroun_B_transformed)
y_france.to_csv('../data/processed/y_france.csv', index=False)
y_cameroun.to_csv('../data/processed/y_cameroun.csv', index=False)

print("✅ Pipelines et matrices sauvegardés.")
print("\nFeature names Modèle A après encoding:")
print(get_feature_names(preprocessor_A))


✅ Pipelines et matrices sauvegardés.

Feature names Modèle A après encoding:
['num__age_maternel', 'num__imc_ordinal', 'num__tension_ordinal', 'num__sa_premiere_consult', 'cat__parite_Multipare_2', 'cat__parite_Multipare_3', 'cat__parite_Nullipare', 'cat__parite_Primipare', 'cat__atcd_familial_diabete_1er_deg_Non', 'cat__atcd_familial_diabete_1er_deg_Oui', 'cat__atcd_gdm_Non', 'cat__atcd_gdm_Oui', 'cat__atcd_macrosomie_Non', 'cat__atcd_macrosomie_Oui', 'cat__sopk_Non', 'cat__sopk_Oui', 'cat__sedentarite_Non', 'cat__sedentarite_Oui', 'cat__tabagisme_Non', 'cat__tabagisme_Oui', 'cat__hta_ou_preeclampsie_Non', 'cat__hta_ou_preeclampsie_Oui']


➡️ **Suite : notebook 04_model_training.ipynb**